# Spam Email Classifier - Minor Project

Machine Learning with Python

**Objective:** classify emails/messages as spam or not spam (ham).

I am using the SMS Spam Collection dataset from UCI (same one that is on Kaggle). It has 5572 labelled messages.

Steps I followed:
1. Load the data
2. Explore it a bit
3. Clean/preprocess the text
4. Convert to TF-IDF features
5. Train Naive Bayes, Logistic Regression and SVM
6. Compare them and tune the best one

In [ ]:
import pandas as pd
import numpy as np
import string
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# nltk.download('stopwords')   # run this once if you dont have it

## 1. Load the dataset

The file is tab separated, first column is the label (ham/spam), second is the message. There is no header row.

In [ ]:
df = pd.read_csv('data/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
df.head()

In [ ]:
print('Shape:', df.shape)
df['label'].value_counts()

So the data is imbalanced - way more ham (4825) than spam (747). Around 13% is spam. Good to keep in mind, thats why accuracy alone is not enough and we look at precision/recall/f1 too.

In [ ]:
# quick bar chart of the class distribution
df['label'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Ham vs Spam count')
plt.ylabel('number of messages')
plt.show()

In [ ]:
# is there a length difference between spam and ham? spam is usually longer
df['length'] = df['message'].apply(len)
df.groupby('label')['length'].mean()

Yes spam messages are longer on average (138 chars vs 71). Makes sense, spam tries to cram in offers and links.

## 2. Preprocess the text

For each message I: lowercase it, remove punctuation, remove stopwords (like 'the', 'is', 'a') and then stem the words (running -> run) so similar words become one.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(msg):
    msg = msg.lower()
    msg = msg.translate(str.maketrans('', '', string.punctuation))
    words = []
    for w in msg.split():
        if w not in stop_words:
            words.append(stemmer.stem(w))
    return ' '.join(words)

# test it on one message
print('BEFORE:', df['message'][2])
print('AFTER :', clean_text(df['message'][2]))

In [ ]:
df['cleaned'] = df['message'].apply(clean_text)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df[['message', 'cleaned', 'target']].head()

## 3. TF-IDF features

TF-IDF turns the text into numbers. It gives higher weight to words that are important in a message but rare across all messages. I limited it to the top 3000 words so it doesnt get too big.

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['cleaned'])
y = df['target']
print('Feature matrix shape:', X.shape)

## 4. Train/test split

80% train, 20% test. Using stratify so the spam ratio is the same in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape[0], ' Test:', X_test.shape[0])

## 5. Train and compare the 3 models

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (Linear)': LinearSVC()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('====', name, '====')
    print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
    print('Precision:', round(precision_score(y_test, y_pred), 4))
    print('Recall   :', round(recall_score(y_test, y_pred), 4))
    print('F1-score :', round(f1_score(y_test, y_pred), 4))
    print()

I thought Naive Bayes would win since most spam tutorials use it, but the SVM actually got the best F1 score here. Naive Bayes has perfect precision (never marks a real message as spam) but its recall is lower so it misses some spam. SVM has the best balance.

## 6. Tune the SVM

Using GridSearchCV to find the best value of C (the regularization parameter).

In [ ]:
params = {'C': [0.1, 0.5, 1.0, 2.0, 5.0]}
grid = GridSearchCV(LinearSVC(), params, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print('Best C:', grid.best_params_['C'])

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

In [ ]:
# confusion matrix of the final model
cm = confusion_matrix(y_test, y_pred)
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix - tuned SVM')
plt.colorbar()
plt.xticks([0, 1], ['ham', 'spam'])
plt.yticks([0, 1], ['ham', 'spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.show()

## 7. Test on new messages

In [ ]:
def predict_message(msg):
    vec = tfidf.transform([clean_text(msg)])
    return 'SPAM' if best_model.predict(vec)[0] == 1 else 'HAM'

print(predict_message('Congratulations! You won a free iPhone, click the link to claim'))
print(predict_message('Hey can you call me when you are free'))
print(predict_message('URGENT your account is suspended verify now'))

## Conclusion

The tuned Linear SVM gave the best result with about **98-99% accuracy** and an **F1 score of ~0.95** for the spam class.

What I learned:
- Text needs a lot of cleaning before the model can use it
- TF-IDF is a simple but effective way to turn text into features
- On imbalanced data, accuracy can look high even for a bad model, so precision/recall/F1 matter more
- Naive Bayes is fast and never flagged a real message as spam, but SVM caught more of the actual spam

Possible improvements: try n-grams, use a bigger email dataset, or try deep learning models.